# Apples-to-Apples Evaluation Attempt — Official Split + Frame-Level AP

**What this is for:** published XD-Violence results report frame-level
Average Precision (AP), evaluated on the 800 official test videos, with
models trained only on the official train split. Every result so far in
this project (`docs/findings.md`) used a custom random 80/20 resample
across a pooled riot+normal set, and reported clip-level accuracy -- not
directly comparable, as documented in `docs/context.md`.

**What this notebook actually closes, and what it doesn't:**
- CLOSES: uses the dataset's real official train/test split (not a
  resample), and computes the real frame-level AP metric against the
  dataset's own frame-level ground truth (`annotations.txt`) -- same
  metric, same official test files as published work.
- DOES NOT CLOSE: still Riot-only vs. Normal (not all 6 XD-Violence
  violence classes lumped as "abnormal"), and uses a **flat per-video
  broadcast score** (every frame of a clip gets that clip's one predicted
  probability) rather than true temporal localization within a video.

**Honest framing for presenting this number:** it is a real AP score,
computed the same way published papers compute theirs, on the same
official test files -- but on a narrower class scope and without
temporal localization. State both facts together, every time this number
is mentioned. Do not present it as *the* XD-Violence benchmark number.

**Requires:** `feature_extraction_fusion.ipynb` already run (any branch
combination currently on Drive is used -- whatever `features_*.csv`
files exist), and the riot/normal clips still present on Drive (frame
counts are probed directly from the video files).

## Step 0 — Setup

In [ ]:
!pip install scikit-learn requests opencv-python-headless -q

In [ ]:
import glob
import os
import re

import cv2
import numpy as np
import pandas as pd
import requests
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import average_precision_score
from google.colab import drive

drive.mount('/content/drive')

RIOT_DIR = '/content/drive/MyDrive/CVPHG/xdviolence_riot'
NORMAL_DIR = '/content/drive/MyDrive/CVPHG/xdviolence_normal'
FEATURES_DIR = '/content/drive/MyDrive/CVPHG/fusion_features'
SEED = 42

## Step 1 — Load features and split by the OFFICIAL train/test column

Not a random resample this time -- `split` comes from which HF directory
each clip was listed under (`test_videos` = official test, everything
else = official train). This is the fix identified while discussing
comparability.

In [ ]:
csv_paths = sorted(glob.glob(os.path.join(FEATURES_DIR, 'features_*.csv')))
print(f"Using {len(csv_paths)} feature file(s): {[os.path.basename(p) for p in csv_paths]}")

dfs = [pd.read_csv(p) for p in csv_paths]
df = dfs[0]
for other in dfs[1:]:
    new_cols = [c for c in other.columns if c not in df.columns]
    df = df.merge(other[['video_id'] + new_cols], on='video_id', how='outer')
df['label'] = df['label'].fillna(method='ffill').fillna(method='bfill')
df = df.dropna()

feature_cols = [c for c in df.columns if c not in ('video_id', 'label', 'split')]
print(f"{len(df)} clips, {len(feature_cols)} feature dimensions")

train_df = df[df['split'] == 'train'].reset_index(drop=True)
test_df = df[df['split'] == 'test'].reset_index(drop=True)
print(f"OFFICIAL train: {len(train_df)} ({(train_df['label']=='riot').sum()} riot / {(train_df['label']=='normal').sum()} normal)")
print(f"OFFICIAL test:  {len(test_df)} ({(test_df['label']=='riot').sum()} riot / {(test_df['label']=='normal').sum()} normal)")

## Step 2 — Train on the official train split, predict probabilities for the official test split

In [ ]:
X_train = train_df[feature_cols].values
y_train = (train_df['label'] == 'riot').astype(int).values
X_test = test_df[feature_cols].values

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

mlp = MLPClassifier(
    hidden_layer_sizes=(256, 64), activation='relu', alpha=1e-4,
    early_stopping=True, n_iter_no_change=15, max_iter=500, random_state=SEED,
)
mlp.fit(X_train_s, y_train)
test_df = test_df.copy()
test_df['riot_proba'] = mlp.predict_proba(X_test_s)[:, 1]
print(f"Trained on {len(X_train)} official-train clips. Predicted probabilities for {len(test_df)} official-test clips.")

## Step 3 — Pull frame-level ground truth and build the flat-broadcast frame arrays

`annotations.txt` gives anomalous frame intervals **local to each clip
file** (frame 0 = this clip's first frame, not the original source
video's) -- verified directly against the file before writing this (see
`docs/findings.md`, 2026-09-18 entry). Normal clips have no entry -- every
frame of a Normal clip is a negative, by construction (no interval to
look up).

In [ ]:
ANNOTATIONS_URL = "https://roc-ng.github.io/XD-Violence/images/annotations.txt"
resp = requests.get(ANNOTATIONS_URL, timeout=60)
resp.raise_for_status()

anomaly_intervals = {}  # video_id (no extension) -> list of (start, end) frame pairs
for line in resp.text.splitlines():
    line = line.strip()
    if not line:
        continue
    parts = line.split()
    vid_base = parts[0]
    nums = [int(x) for x in parts[1:]]
    intervals = [(nums[i], nums[i + 1]) for i in range(0, len(nums) - 1, 2)]
    anomaly_intervals[vid_base] = intervals

print(f"Parsed {len(anomaly_intervals)} annotated (anomalous) test videos from annotations.txt")

In [ ]:
def get_frame_count(video_path):
    cap = cv2.VideoCapture(video_path)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return n


all_y_true = []
all_y_score = []
matched_riot = 0
unmatched_riot = []

for row in test_df.to_dict('records'):
    clip_dir = RIOT_DIR if row['label'] == 'riot' else NORMAL_DIR
    video_path = os.path.join(clip_dir, row['video_id'])
    n_frames = get_frame_count(video_path)
    if n_frames <= 0:
        continue

    y_true = np.zeros(n_frames, dtype=int)
    if row['label'] == 'riot':
        vid_base = os.path.splitext(row['video_id'])[0]
        intervals = anomaly_intervals.get(vid_base)
        if intervals is None:
            unmatched_riot.append(row['video_id'])
        else:
            matched_riot += 1
            for start, end in intervals:
                y_true[max(0, start):min(n_frames, end + 1)] = 1

    y_score = np.full(n_frames, row['riot_proba'])
    all_y_true.append(y_true)
    all_y_score.append(y_score)

print(f"Riot test clips matched to an annotation entry: {matched_riot}/{(test_df['label']=='riot').sum()}")
if unmatched_riot:
    print(f"Unmatched riot clip IDs (treated as all-negative, which UNDERSTATES AP -- flag if this list is large):")
    for vid in unmatched_riot[:10]:
        print(' ', vid)
    if len(unmatched_riot) > 10:
        print(f"  ... and {len(unmatched_riot) - 10} more")

y_true_concat = np.concatenate(all_y_true)
y_score_concat = np.concatenate(all_y_score)
print(f"\nTotal frames across all official test clips: {len(y_true_concat)} ({y_true_concat.sum()} anomalous)")

## Step 4 — Compute frame-level AP

In [ ]:
ap = average_precision_score(y_true_concat, y_score_concat)
print(f"Frame-level AP (Riot-only vs. Normal, official split, flat broadcast score): {ap:.4f} ({ap:.2%})")
print()
print("Compare against published full-task numbers (all 6 classes lumped, true")
print("temporal localization, weakly-supervised): DSANet 86.95%, ReFLIP 85.81%,")
print("TCVADS 85.58%, VadCLIP 84.51% -- NOT the same task, see notebook header")
print("before quoting this number next to those.")

## Next steps

- If `unmatched_riot` is non-empty, those clips contribute all-negative
  ground truth, which *understates* this AP number -- worth checking a
  few by hand if the list is more than a handful.
- Save this result (`ap`, `matched_riot`, clip counts) into
  `results/` and log it in `docs/findings.md` once reviewed, same as
  every other result in this project -- tag the entry clearly with what
  this closes (metric, official split) and what it doesn't (class scope,
  temporal localization), per `docs/context.md`'s comparability section.
- The full rigorous match (6 classes lumped, weakly-supervised MIL
  training) is still a separate, bigger undertaking -- this notebook does
  not attempt it.